In [1]:
!nvidia-smi

Wed Mar  4 12:52:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q --upgrade pip
!pip install -q transformers accelerate datasets peft bitsandbytes sentencepiece safetensors huggingface_hub
!pip install -q tiktoken

In [3]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import json

eval_prompts = [
  {"prompt": "Explain Return on Equity (ROE)."},
  {"prompt": "Why do bond prices fall when interest rates rise?"},
  {"prompt": "What does negative free cash flow indicate?"},
  {"prompt": "Summarize: Revenue grew 15% but margins declined due to rising costs."},
  {"prompt": "How does inflation affect stock valuations?"},
  {"prompt": "Explain operating leverage."},
  {"prompt": "If EPS is 5 and price is 125, calculate P/E."},
  {"prompt": "Why might a company refinance debt?"},
  {"prompt": "What is the difference between EBITDA and net income?"},
  {"prompt": "How does diversification reduce portfolio risk?"}
]

with open("eval_prompts.json", "w") as f:
    json.dump(eval_prompts, f, indent=2)

print("Evaluation prompts saved.")

Evaluation prompts saved.


In [7]:
# Base Model Evaluation
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

def generate(prompt):
    text = f"### Instruction:\n{prompt}\n\n### Input:\n\n### Response:\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

with open("eval_prompts.json") as f:
    prompts = json.load(f)

base_results = []

for p in prompts:
    response = generate(p["prompt"])
    print("\nPrompt:", p["prompt"])
    print(response)
    print("-"*80)
    base_results.append({"prompt": p["prompt"], "response": response})

with open("base_results.json", "w") as f:
    json.dump(base_results, f, indent=2)

print("Base model results saved.")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...


Prompt: If EPS is 5 and price is 125, calculate P/E.
### Instruction:
If EPS is 5 and price is 125, calculate P/E.

### Input:

### Response:
25

### Explanation:
The formula to calculate P/E is P/E = Price / EPS. In this case, P/E = 125 / 5 = 25. 

### Code:
```
EPS = 5
price = 125
P_E = price / EPS
print(P_E)
``` 

### Output:
25.0
### End Instruction.
--------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: Why might a company refinance debt?
### Instruction:
Why might a company refinance debt?

### Input:

### Response:
A company might refinance debt for several reasons:

1. **Lower interest rates**: If interest rates have fallen since the original loan was taken out, refinancing can help the company save money on interest payments.
2. **Longer repayment period**: Refinancing can provide a longer repayment period, which can reduce the monthly payment burden on the company.
3. **Reduced debt burden**: Refinancing can help the company consolidate multiple debts into a single loan with a lower interest rate, reducing the overall debt burden.
4. **Access to additional funds**: Refinancing can provide access to additional funds, which can be used for expansion, modernization, or other business purposes.
5. **Improved cash flow**: Refinancing can help
--------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: What is the difference between EBITDA and net income?
### Instruction:
What is the difference between EBITDA and net income?

### Input:

### Response:
EBITDA (Earnings Before Interest, Taxes, Depreciation, and Amortization) and net income are two financial metrics used to measure a company's profitability. The main difference between the two is that EBITDA excludes certain non-operating items, such as interest expenses, taxes, depreciation, and amortization, whereas net income includes all these items.

Net income, also known as net earnings or profit, is the company's total earnings or profit after all expenses, including interest, taxes, depreciation, and amortization, have been deducted from its total revenue.

EBITDA, on the other hand, is a measure of a company's core profitability, excluding the above-mentioned non-operating items. It is calculated by adding
--------------------------------------------------------------------------------

Prompt: How does diversificatio

In [5]:
#LORA Model Evaluation

import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"
LORA_PATH = "/content/drive/MyDrive/lora_finance"

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()

def generate(prompt):
    text = f"### Instruction:\n{prompt}\n\n### Input:\n\n### Response:\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

with open("eval_prompts.json") as f:
    prompts = json.load(f)

lora_results = []

for p in prompts:
    response = generate(p["prompt"])
    print("\nPrompt:", p["prompt"])
    print(response)
    print("-"*80)
    lora_results.append({"prompt": p["prompt"], "response": response})

with open("lora_results.json", "w") as f:
    json.dump(lora_results, f, indent=2)

print("LoRA model results saved.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: Explain Return on Equity (ROE).
### Instruction:
Explain Return on Equity (ROE).

### Input:

### Response:
Return on Equity (ROE) measures how efficiently a company generates profit from shareholders' equity. It is calculated as net income divided by shareholders' equity.
--------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: Why do bond prices fall when interest rates rise?
### Instruction:
Why do bond prices fall when interest rates rise?

### Input:

### Response:
Bond prices fall when interest rates rise because existing bonds with lower coupons become less attractive compared to newly issued bonds offering higher yields.
--------------------------------------------------------------------------------


...


Prompt: If EPS is 5 and price is 125, calculate P/E.
### Instruction:
If EPS is 5 and price is 125, calculate P/E.

### Input:

### Response:
The P/E ratio is 25.
--------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: Why might a company refinance debt?
### Instruction:
Why might a company refinance debt?

### Input:

### Response:
A company may refinance debt to lower interest payments or extend maturity dates.
--------------------------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Prompt: What is the difference between EBITDA and net income?
### Instruction:
What is the difference between EBITDA and net income?

### Input:

### Response:
EBITDA excludes interest and tax expenses, while net income includes these expenses.
--------------------------------------------------------------------------------

Prompt: How does diversification reduce portfolio risk?
### Instruction:
How does diversification reduce portfolio risk?

### Input:

### Response:
Diversification reduces risk by spreading investments across different assets, minimizing the impact of any single asset’s poor performance.
--------------------------------------------------------------------------------
LoRA model results saved.
